# POI Category Alignment: Sweden vs US

Compares POI categories between:
- **Sweden**: SafeGraph POI data with GPS-based visit assignments
- **US**: SafeGraph weekly_patterns foot traffic data

Both datasets come from SafeGraph, so categories should align. This notebook identifies:
1. Category distribution differences (TOP_CATEGORY, SUB_CATEGORY)
2. Missing or mismatched categories
3. Visit distribution patterns
4. Recommendations for unified category schema

In [42]:
%cd /workspace

/workspace


In [43]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Paths
SE_POI_DIR = Path("dbs/poi_se/POI_se_2026_03_05")
SE_ASSIGNED = Path("dbs/poi_assignment/stops_poi_assigned.parquet")
US_WP_DIR = Path("dbs/us_foot_traffic/weekly_patterns")

print(f"Swedish POI: {SE_POI_DIR}")
print(f"Swedish Assigned: {SE_ASSIGNED}")
print(f"US Weekly Patterns: {US_WP_DIR}")

Swedish POI: dbs/poi_se/POI_se_2026_03_05
Swedish Assigned: dbs/poi_assignment/stops_poi_assigned.parquet
US Weekly Patterns: dbs/us_foot_traffic/weekly_patterns


## 1. Load Swedish POI Data

In [44]:
# Load all Swedish POI files
se_poi_files = sorted(SE_POI_DIR.glob("*.parquet"))
print(f"Swedish POI files: {len(se_poi_files)}")

df_se_poi = pd.concat([pd.read_parquet(f) for f in se_poi_files], ignore_index=True)
print(f"Total Swedish POIs: {len(df_se_poi):,}")

# Strip whitespace from category columns (for consistency)
if 'TOP_CATEGORY' in df_se_poi.columns:
    df_se_poi['TOP_CATEGORY'] = df_se_poi['TOP_CATEGORY'].str.strip()
if 'SUB_CATEGORY' in df_se_poi.columns:
    df_se_poi['SUB_CATEGORY'] = df_se_poi['SUB_CATEGORY'].str.strip()
print("Stripped whitespace from category columns")

print(f"\nColumns: {list(df_se_poi.columns)}")

Swedish POI files: 32
Total Swedish POIs: 658,731
Stripped whitespace from category columns

Columns: ['BRANDS', 'CATEGORY_TAGS', 'CITY', 'CLOSED_ON', 'DOMAINS', 'ENCLOSED', 'GEOMETRY_TYPE', 'INCLUDES_PARKING_LOT', 'ISO_COUNTRY_CODE', 'IS_SYNTHETIC', 'LATITUDE', 'LOCATION_NAME', 'LONGITUDE', 'NAICS_CODE', 'OPENED_ON', 'OPEN_HOURS', 'PARENT_PLACEKEY', 'PHONE_NUMBER', 'PLACEKEY', 'POLYGON_CLASS', 'POLYGON_WKT', 'POSTAL_CODE', 'REGION', 'STORE_ID', 'STREET_ADDRESS', 'SUB_CATEGORY', 'TOP_CATEGORY', 'TRACKING_CLOSED_SINCE', 'WEBSITE', 'WKT_AREA_SQ_METERS']


In [45]:
# Check column names (SafeGraph uses UPPERCASE)
df_se_poi.head(3)

,BRANDS,CATEGORY_TAGS,CITY,CLOSED_ON,DOMAINS,ENCLOSED,GEOMETRY_TYPE,INCLUDES_PARKING_LOT,ISO_COUNTRY_CODE,IS_SYNTHETIC,LATITUDE,LOCATION_NAME,LONGITUDE,NAICS_CODE,OPENED_ON,OPEN_HOURS,PARENT_PLACEKEY,PHONE_NUMBER,PLACEKEY,POLYGON_CLASS,POLYGON_WKT,POSTAL_CODE,REGION,STORE_ID,STREET_ADDRESS,SUB_CATEGORY,TOP_CATEGORY,TRACKING_CLOSED_SINCE,WEBSITE,WKT_AREA_SQ_METERS
0,[],"[""Contractors""]",Dingle,None,[],False,POLYGON,False,SE,False,58.692139,Johansson Bengt Arne,11.611203,561730,None,NaN,NaN,+4630316768,zzy-222@33k-mk4-brk,OWNED_POLYGON,POLYGON ((11.611308452465277 58.69222820463569...,455 97,Västra Götaland County,NaN,Björnåsen 1,Landscaping Services,Services to Buildings and Dwellings,2019-07-01,NaN,145
1,[],"[""Accessories"",""Boutique"",""Clothing""]",Trelleborg,None,[],False,POLYGON,False,SE,False,55.378228,Denniz Konsult,13.151666,448120,None,NaN,NaN,+464043690,zzy-222@524-d8p-yqf,OWNED_POLYGON,POLYGON ((13.151533786079106 55.37821890009666...,231 63,Skåne County,NaN,Flygelgränd 26,Women's Clothing Stores,Clothing Stores,2019-07-01,NaN,135
2,[],[],Lomma,None,[],False,POLYGON,False,SE,False,55.668962,John A. Jonsson,13.072857,512110,None,NaN,NaN,+4640414591,zzy-222@524-kfx-q4v,OWNED_POLYGON,POLYGON ((13.072787508639097 55.66902961215978...,234 32,Skåne County,NaN,Allégatan 18,Motion Picture and Video Production,Motion Picture and Video Industries,2019-07-01,NaN,176


In [46]:
# Identify category columns
cat_cols_se = [c for c in df_se_poi.columns if 'CATEGORY' in c.upper() or 'NAICS' in c.upper()]
print(f"Category columns: {cat_cols_se}")

for col in cat_cols_se:
    print(f"\n{col}: {df_se_poi[col].nunique()} unique values")
    print(df_se_poi[col].value_counts().head(10))

Category columns: ['CATEGORY_TAGS', 'NAICS_CODE', 'SUB_CATEGORY', 'TOP_CATEGORY']

CATEGORY_TAGS: 9413 unique values
CATEGORY_TAGS
[]                                                      308875
["Property Management"]                                  40180
["Contractors"]                                          34457
["Auto Parts & Accessories","Car Dealer","Car Wash"]     13966
["Hair Salon"]                                            9178
["Graphic Design"]                                        7750
["Painters"]                                              7014
["Accessories","Boutique","Clothing"]                     6774
["HVAC"]                                                  6774
["Electricians"]                                          6544
Name: count, dtype: int64

NAICS_CODE: 536 unique values
NAICS_CODE
485119    49392
531311    44471
485113    40005
561730    28134
722511    23265
236118    20443
624410    15038
812112    15004
441120    13485
238220    12152
Name: count

## 2. Load Swedish POI Assignment Results

In [47]:
# Load assigned stops - ONLY poi_id column (memory optimized)
if SE_ASSIGNED.exists():
    # Only load the columns we need
    df_se_assigned = pd.read_parquet(SE_ASSIGNED, columns=['poi_id'])
    print(f"Swedish assigned stops: {len(df_se_assigned):,}")
    print(f"Columns loaded: {list(df_se_assigned.columns)}")
    print(f"Memory usage: {df_se_assigned.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
else:
    print(f"File not found: {SE_ASSIGNED}")
    df_se_assigned = None

Swedish assigned stops: 175,487,188
Columns loaded: ['poi_id']
Memory usage: 2824.3 MB


In [48]:
# Check assignment results
if df_se_assigned is not None:
    print("Assignment overview:")
    if 'poi_id' in df_se_assigned.columns:
        matched = df_se_assigned['poi_id'].notna().sum()
        print(f"  Matched to POI: {matched:,} ({matched/len(df_se_assigned)*100:.1f}%)")
    if 'poi_match_tier' in df_se_assigned.columns:
        print(f"\nMatch tier distribution:")
        print(df_se_assigned['poi_match_tier'].value_counts())

Assignment overview:
  Matched to POI: 80,820,135 (46.1%)


In [49]:
# Merge with POI categories to get category for each visit
if df_se_assigned is not None:
    # Identify POI ID column in POI data
    poi_id_col = 'PLACEKEY' if 'PLACEKEY' in df_se_poi.columns else 'ID' if 'ID' in df_se_poi.columns else df_se_poi.columns[0]
    print(f"POI ID column: {poi_id_col}")
    
    # Find category columns
    top_cat_col = [c for c in df_se_poi.columns if 'TOP' in c.upper() and 'CATEGORY' in c.upper()]
    sub_cat_col = [c for c in df_se_poi.columns if 'SUB' in c.upper() and 'CATEGORY' in c.upper()]
    
    top_cat_col = top_cat_col[0] if top_cat_col else None
    sub_cat_col = sub_cat_col[0] if sub_cat_col else None
    
    print(f"TOP_CATEGORY column: {top_cat_col}")
    print(f"SUB_CATEGORY column: {sub_cat_col}")

POI ID column: PLACEKEY
TOP_CATEGORY column: TOP_CATEGORY
SUB_CATEGORY column: SUB_CATEGORY


## 3. Load US Weekly Patterns Data

In [50]:
# Load US weekly patterns (sample for exploration)
us_wp_files = sorted(US_WP_DIR.glob("*.parquet"))
print(f"US weekly patterns files: {len(us_wp_files)}")

if us_wp_files:
    # Check file sizes to ensure they're valid
    valid_files = [f for f in us_wp_files if f.stat().st_size > 1000000]  # > 1MB
    print(f"Valid files (>1MB): {len(valid_files)}")
    
    if valid_files:
        # Load first file as sample
        df_us_sample = pd.read_parquet(valid_files[0])
        print(f"\nSample file: {valid_files[0].name}")
        print(f"Rows: {len(df_us_sample):,}")
        print(f"Columns: {list(df_us_sample.columns)}")
else:
    print("No US weekly patterns files found")
    df_us_sample = None

US weekly patterns files: 40
Valid files (>1MB): 40

Sample file: 2024-03-04--data_01c29bf5-0107-dbc6-0042-fa070657a816_004_4_0.snappy.parquet
Rows: 527,327
Columns: ['BRAND', 'BUCKETED_DWELL_TIMES', 'CITY', 'CLOSE_DATE', 'DATE_RANGE_END', 'DATE_RANGE_START', 'DEVICE_TYPE', 'DISTANCE_FROM_HOME', 'FOOTPRINT_ID', 'ID_STORE', 'ISO_COUNTRY_CODE', 'IS_DISTRIBUTOR', 'LATITUDE', 'LOCATION_NAME', 'LONGITUDE', 'MEDIAN_DWELL', 'MSA_CODE', 'NAICS_CODE', 'OPEN_DATE', 'PERSISTENT_ID', 'PERSISTENT_ID_STORE', 'POI_CBG', 'POSTAL_CODE', 'REGION', 'RELATED_SAME_DAY_BRAND', 'RELATED_SAME_WEEK_BRAND', 'STREET_ADDRESS', 'SUB_CATEGORY', 'TICKER', 'TOP_CATEGORY', 'VISITOR_COUNTRY_OF_ORIGIN', 'VISITOR_COUNTS', 'VISITOR_DAYTIME_CBGS', 'VISITOR_HOME_AGGREGATION', 'VISITOR_HOME_CBGS', 'VISITS_BY_DAY', 'VISITS_BY_EACH_HOUR', 'VISIT_COUNTS']


In [51]:
df_us_sample.head()

,BRAND,BUCKETED_DWELL_TIMES,CITY,CLOSE_DATE,DATE_RANGE_END,DATE_RANGE_START,DEVICE_TYPE,DISTANCE_FROM_HOME,FOOTPRINT_ID,ID_STORE,ISO_COUNTRY_CODE,IS_DISTRIBUTOR,LATITUDE,LOCATION_NAME,LONGITUDE,MEDIAN_DWELL,MSA_CODE,NAICS_CODE,OPEN_DATE,PERSISTENT_ID,PERSISTENT_ID_STORE,POI_CBG,POSTAL_CODE,REGION,RELATED_SAME_DAY_BRAND,RELATED_SAME_WEEK_BRAND,STREET_ADDRESS,SUB_CATEGORY,TICKER,TOP_CATEGORY,VISITOR_COUNTRY_OF_ORIGIN,VISITOR_COUNTS,VISITOR_DAYTIME_CBGS,VISITOR_HOME_AGGREGATION,VISITOR_HOME_CBGS,VISITS_BY_DAY,VISITS_BY_EACH_HOUR,VISIT_COUNTS
0,All Other Specialty Food Stores,NaN,Bremerton,2038-01-01,2024-03-10 00:00:00+00:00,2024-03-04 00:00:00+00:00,NaN,NaN,5014245170173332718,8646549,US,False,47.582218,All Other Specialty Food Stores,-122.608063,NaN,14740,445299,1970-01-01,A1_NAICS-445299,8646549,530350804003,98310,WA,NaN,NaN,2340 Trenton Ave Ne,All Other Specialty Food Stores,NAICS-445299,Specialty Food Stores,NaN,14,NaN,NaN,NaN,"[0,0,14,0,0,14,0]",NaN,27
1,Wireless Telecommunications Carriers (except S...,NaN,Wisconsin Dells,2038-01-01,2024-03-10 00:00:00+00:00,2024-03-04 00:00:00+00:00,NaN,NaN,8261614713950411849,11433104,US,False,43.601289,Wireless Telecommunications Carriers (except S...,-89.792853,NaN,12660,517312,1970-01-01,A1_NAICS-517112,11433104,551110001041,53965,WI,"{""Jersey Mike's"":14}","{""American Airlines"":14,""Event Venues"":14,""Hot...",1010 Wisconsin Dells Pkwy,Wireless Telecommunications Carriers (except S...,NAICS-517112,Wired and Wireless Telecommunications Carriers,NaN,14,NaN,NaN,NaN,"[0,14,0,0,0,0,0]",NaN,14
2,All Other Home Furnishings Stores,NaN,Vienna,2038-01-01,2024-03-10 00:00:00+00:00,2024-03-04 00:00:00+00:00,NaN,NaN,1856774267155105828,8375694,US,False,41.221279,All Other Home Furnishings Stores,-80.636498,NaN,49660,442299,1970-01-01,A1_NAICS-442299,8375694,391559312002,44473,OH,NaN,NaN,824 Sodom Hutchings Rd Se,All Other Home Furnishings Stores,NAICS-442299,Home Furnishings Stores,NaN,14,"{""391559312002"":14}","{""39155931200"":14}","{""391559312002"":14}","[0,14,14,14,0,0,14]",NaN,55
3,Electronics Stores,NaN,San Diego,2038-01-01,2024-03-10 00:00:00+00:00,2024-03-04 00:00:00+00:00,NaN,NaN,8188852089708494234,7778520,US,False,32.901871,Electronics Stores,-117.207878,NaN,41740,443142,1970-01-01,A1_NAICS-443142,7778520,060730083462,92121,CA,NaN,NaN,10182 Telesis Ct,Electronics Stores,NAICS-443142,Electronics and Appliance Stores,NaN,14,NaN,NaN,NaN,"[14,14,14,14,14,14,0]",NaN,82
4,USPS,NaN,Beersheba Springs,2038-01-01,2024-03-10 00:00:00+00:00,2024-03-04 00:00:00+00:00,NaN,NaN,8858681014763741166,4839094,US,False,35.465424,United States Postal Service,-85.661734,NaN,None,491110,1970-01-01,A1_USPS-A,4839094,470619550001,37305-9998,TN,NaN,NaN,19472 State Route 56,Postal Service,USPS-A,Postal Service,NaN,14,"{""470619550001"":14}","{""47061955000"":14}","{""470619550001"":14}","[14,14,14,14,14,0,0]",NaN,68


In [52]:
# Check US data structure
if df_us_sample is not None:
    df_us_sample.head(3)

In [53]:
# Load US files for category analysis (MEMORY-OPTIMIZED)
# Only need unique POIs for category distribution - don't need all POI-weeks
if valid_files:
    print(f"Loading US files for category analysis...")
    
    # Only load category columns (minimal memory)
    cat_cols_us = ['ID_STORE', 'TOP_CATEGORY', 'SUB_CATEGORY', 'NAICS_CODE', 'VISIT_COUNTS']
    
    # Load just first 5 files (enough for category distribution)
    files_to_load = valid_files[:5]
    print(f"  Loading {len(files_to_load)} files (sample for category analysis)")
    
    dfs = []
    for i, f in enumerate(files_to_load):
        df_temp = pd.read_parquet(f)
        # Only keep columns that exist
        cols_exist = [c for c in cat_cols_us if c in df_temp.columns]
        dfs.append(df_temp[cols_exist])
        print(f"    Loaded {i+1}/{len(files_to_load)}: {len(df_temp):,} rows")
        del df_temp
    
    df_us_raw = pd.concat(dfs, ignore_index=True)
    del dfs
    print(f"  Total rows loaded: {len(df_us_raw):,}")
    
    # FIX: Strip whitespace from category columns (US data has trailing spaces!)
    print(f"  Stripping whitespace from category columns...")
    df_us_raw['TOP_CATEGORY'] = df_us_raw['TOP_CATEGORY'].str.strip()
    df_us_raw['SUB_CATEGORY'] = df_us_raw['SUB_CATEGORY'].str.strip()
    
    # Get unique POIs only (this is what we need for category distribution)
    df_us_poi = df_us_raw.drop_duplicates(subset=['ID_STORE']).copy()
    print(f"  Unique US POIs: {len(df_us_poi):,}")
    
    # Keep df_us for visit analysis but make it smaller
    if 'VISIT_COUNTS' in df_us_raw.columns:
        df_us = df_us_raw[['ID_STORE', 'TOP_CATEGORY', 'SUB_CATEGORY', 'VISIT_COUNTS']].copy()
    else:
        df_us = df_us_raw[['ID_STORE', 'TOP_CATEGORY', 'SUB_CATEGORY']].copy()
    
    # Clean up
    del df_us_raw
    import gc; gc.collect()
    print(f"  Memory cleaned up")

Loading US files for category analysis...
  Loading 5 files (sample for category analysis)
    Loaded 1/5: 527,327 rows
    Loaded 2/5: 532,399 rows
    Loaded 3/5: 533,576 rows
    Loaded 4/5: 526,116 rows
    Loaded 5/5: 541,614 rows
  Total rows loaded: 2,661,032
  Stripping whitespace from category columns...
  Unique US POIs: 2,661,032
  Memory cleaned up


## 4. Compare TOP_CATEGORY Distribution

In [54]:
# Swedish TOP_CATEGORY distribution
if top_cat_col and top_cat_col in df_se_poi.columns:
    se_top_dist = df_se_poi[top_cat_col].value_counts(normalize=True).reset_index()
    se_top_dist.columns = ['category', 'se_pct']
    print(f"Swedish TOP_CATEGORY ({df_se_poi[top_cat_col].nunique()} categories):")
    print(se_top_dist.head(20).to_string(index=False))

Swedish TOP_CATEGORY (242 categories):
                                           category   se_pct
                              Urban Transit Systems 0.136087
                  Activities Related to Real Estate 0.067534
                Restaurants and Other Eating Places 0.049747
                             Personal Care Services 0.046907
                Services to Buildings and Dwellings 0.044677
                  Residential Building Construction 0.031078
                     Building Equipment Contractors 0.030037
                                 Automobile Dealers 0.024198
                      Other Schools and Instruction 0.023026
                            Child Day Care Services 0.022829
          Other Amusement and Recreation Industries 0.022218
                                    Clothing Stores 0.019627
            Printing and Related Support Activities 0.019088
                            Home Furnishings Stores 0.016823
                            Other Personal Ser

In [55]:
# US TOP_CATEGORY distribution
if 'df_us_poi' in dir() and 'TOP_CATEGORY' in df_us_poi.columns:
    us_top_dist = df_us_poi['TOP_CATEGORY'].value_counts(normalize=True).reset_index()
    us_top_dist.columns = ['category', 'us_pct']
    print(f"US TOP_CATEGORY ({df_us_poi['TOP_CATEGORY'].nunique()} categories):")
    print(us_top_dist.head(20).to_string(index=False))

US TOP_CATEGORY (235 categories):
                                                              category   us_pct
                                   Restaurants and Other Eating Places 0.164190
                                                     Gasoline Stations 0.060713
                                   Other Miscellaneous Store Retailers 0.052041
                                     Automotive Repair and Maintenance 0.047360
                                                        Grocery Stores 0.038582
                                                Personal Care Services 0.034171
                                       Health and Personal Care Stores 0.029406
                           Activities Related to Credit Intermediation 0.029364
                                                Traveler Accommodation 0.027828
                        Automotive Parts, Accessories, and Tire Stores 0.026643
                                                       Clothing Stores 0.025982
      

In [56]:
# Compare TOP_CATEGORY between Sweden and US
if 'se_top_dist' in dir() and 'us_top_dist' in dir():
    # Merge distributions
    top_compare = pd.merge(se_top_dist, us_top_dist, on='category', how='outer')
    top_compare = top_compare.fillna(0)
    top_compare['diff'] = top_compare['se_pct'] - top_compare['us_pct']
    top_compare['abs_diff'] = top_compare['diff'].abs()
    top_compare = top_compare.sort_values('abs_diff', ascending=False)
    
    print("="*80)
    print("TOP_CATEGORY COMPARISON (sorted by difference)")
    print("="*80)
    print(f"{'Category':<40} {'Sweden %':>10} {'US %':>10} {'Diff':>10}")
    print("-"*80)
    for _, row in top_compare.iterrows():
        print(f"{row['category'][:40]:<40} {row['se_pct']*100:>9.1f}% {row['us_pct']*100:>9.1f}% {row['diff']*100:>+9.1f}%")

TOP_CATEGORY COMPARISON (sorted by difference)
Category                                   Sweden %       US %       Diff
--------------------------------------------------------------------------------
Urban Transit Systems                         13.6%       0.0%     +13.6%
Restaurants and Other Eating Places            5.0%      16.4%     -11.4%
Activities Related to Real Estate              6.8%       0.0%      +6.7%
Gasoline Stations                              1.6%       6.1%      -4.5%
Services to Buildings and Dwellings            4.5%       0.0%      +4.4%
Other Miscellaneous Store Retailers            0.9%       5.2%      -4.3%
Automotive Repair and Maintenance              1.3%       4.7%      -3.5%
Residential Building Construction              3.1%       0.0%      +3.1%
Activities Related to Credit Intermediat       0.1%       2.9%      -2.8%
Grocery Stores                                 1.1%       3.9%      -2.7%
Health and Personal Care Stores                0.5%       

In [57]:
# Categories only in Sweden
if 'top_compare' in dir():
    se_only = top_compare[top_compare['us_pct'] == 0]['category'].tolist()
    us_only = top_compare[top_compare['se_pct'] == 0]['category'].tolist()
    
    print(f"\nCategories ONLY in Sweden ({len(se_only)}):")
    for cat in se_only:
        pct = top_compare[top_compare['category']==cat]['se_pct'].values[0]
        print(f"  - {cat}: {pct*100:.2f}%")
    
    print(f"\nCategories ONLY in US ({len(us_only)}):")
    for cat in us_only:
        pct = top_compare[top_compare['category']==cat]['us_pct'].values[0]
        print(f"  - {cat}: {pct*100:.2f}%")


Categories ONLY in Sweden (59):
  - Urban Transit Systems: 13.61%
  - Child Day Care Services: 2.28%
  - Elementary and Secondary Schools: 1.27%
  - Specialized Design Services: 0.88%
  - Offices of Physicians: 0.81%
  - Transit and Ground Passenger Transportation: 0.69%
  - Offices of Dentists: 0.67%
  - Social Advocacy Organizations: 0.61%
  - Religious Organizations: 0.58%
  - Offices of Other Health Practitioners: 0.58%
  - Outpatient Care Centers: 0.51%
  - Justice, Public Order, and Safety Activities: 0.22%
  - Civic and Social Organizations: 0.21%
  - Continuing Care Retirement Communities and Assisted Living Facilities for the Elderly: 0.13%
  - Individual and Family Services: 0.12%
  - Waste Collection: 0.08%
  - Other Transit and Ground Passenger Transportation: 0.07%
  - Interurban and Rural Bus Transportation: 0.05%
  - Medical and Diagnostic Laboratories: 0.04%
  - Scenic and Sightseeing Transportation, Water: 0.03%
  - Home Health Care Services: 0.02%
  - Charter Bus Ind

## 5. Compare SUB_CATEGORY Distribution

In [58]:
# Swedish SUB_CATEGORY distribution
if sub_cat_col and sub_cat_col in df_se_poi.columns:
    se_sub_dist = df_se_poi[sub_cat_col].value_counts(normalize=True).reset_index()
    se_sub_dist.columns = ['category', 'se_pct']
    print(f"Swedish SUB_CATEGORY ({df_se_poi[sub_cat_col].nunique()} categories):")
    print(se_sub_dist.head(30).to_string(index=False))

Swedish SUB_CATEGORY (478 categories):
                                                        category   se_pct
                                     Other Urban Transit Systems 0.078146
                                   Residential Property Managers 0.070361
                     Bus and Other Motor Vehicle Transit Systems 0.063295
                                            Landscaping Services 0.044513
                                        Full-Service Restaurants 0.036809
                                          Residential Remodelers 0.032344
                                         Child Day Care Services 0.023793
                                                   Beauty Salons 0.023739
                                                Used Car Dealers 0.021336
             Plumbing, Heating, and Air-Conditioning Contractors 0.019227
                         Fitness and Recreational Sports Centers 0.018791
                              Hair, Nail, and Skin Care Services 0.015016

In [59]:
# US SUB_CATEGORY distribution
if 'df_us_poi' in dir() and 'SUB_CATEGORY' in df_us_poi.columns:
    us_sub_dist = df_us_poi['SUB_CATEGORY'].value_counts(normalize=True).reset_index()
    us_sub_dist.columns = ['category', 'us_pct']
    print(f"US SUB_CATEGORY ({df_us_poi['SUB_CATEGORY'].nunique()} categories):")
    print(us_sub_dist.head(30).to_string(index=False))

US SUB_CATEGORY (616 categories):
                                                       category   us_pct
                                       Full-Service Restaurants 0.097421
                                    Limited-Service Restaurants 0.046252
All Other Miscellaneous Store Retailers (except Tobacco Stores) 0.046112
                      Gasoline Stations with Convenience Stores 0.040982
     Supermarkets and Other Grocery (except Convenience) Stores 0.027172
              Other Activities Related to Credit Intermediation 0.024918
                                                  Beauty Salons 0.024675
                                      General Automotive Repair 0.024527
                       Hotels (except Casino Hotels) and Motels 0.024370
            Plumbing, Heating, and Air-Conditioning Contractors 0.024054
                                        Other Gasoline Stations 0.019750
                        Fitness and Recreational Sports Centers 0.019283
                 

In [60]:
# Compare SUB_CATEGORY
if 'se_sub_dist' in dir() and 'us_sub_dist' in dir():
    sub_compare = pd.merge(se_sub_dist, us_sub_dist, on='category', how='outer')
    sub_compare = sub_compare.fillna(0)
    sub_compare['diff'] = sub_compare['se_pct'] - sub_compare['us_pct']
    sub_compare['abs_diff'] = sub_compare['diff'].abs()
    sub_compare = sub_compare.sort_values('abs_diff', ascending=False)
    
    print("="*80)
    print("SUB_CATEGORY COMPARISON (top 40 by difference)")
    print("="*80)
    print(f"{'Category':<45} {'Sweden %':>10} {'US %':>10} {'Diff':>10}")
    print("-"*80)
    for _, row in sub_compare.head(40).iterrows():
        print(f"{str(row['category'])[:45]:<45} {row['se_pct']*100:>9.2f}% {row['us_pct']*100:>9.2f}% {row['diff']*100:>+9.2f}%")

SUB_CATEGORY COMPARISON (top 40 by difference)
Category                                        Sweden %       US %       Diff
--------------------------------------------------------------------------------
Other Urban Transit Systems                        7.81%      0.00%     +7.81%
Residential Property Managers                      7.04%      0.00%     +7.04%
Bus and Other Motor Vehicle Transit Systems        6.33%      0.00%     +6.33%
Full-Service Restaurants                           3.68%      9.74%     -6.06%
All Other Miscellaneous Store Retailers (exce      0.08%      4.61%     -4.53%
Landscaping Services                               4.45%      0.00%     +4.45%
Limited-Service Restaurants                        0.96%      4.63%     -3.66%
Gasoline Stations with Convenience Stores          0.69%      4.10%     -3.41%
Residential Remodelers                             3.23%      0.01%     +3.23%
Other Activities Related to Credit Intermedia      0.07%      2.49%     -2.43%
Chi

In [61]:
# SUB_CATEGORY mismatches
if 'sub_compare' in dir():
    se_only_sub = sub_compare[sub_compare['us_pct'] == 0].sort_values('se_pct', ascending=False)
    us_only_sub = sub_compare[sub_compare['se_pct'] == 0].sort_values('us_pct', ascending=False)
    
    print(f"\nSUB_CATEGORY ONLY in Sweden ({len(se_only_sub)}, showing top 20):")
    for _, row in se_only_sub.head(20).iterrows():
        print(f"  - {row['category']}: {row['se_pct']*100:.2f}%")
    
    print(f"\nSUB_CATEGORY ONLY in US ({len(us_only_sub)}, showing top 20):")
    for _, row in us_only_sub.head(20).iterrows():
        print(f"  - {row['category']}: {row['us_pct']*100:.2f}%")


SUB_CATEGORY ONLY in Sweden (162, showing top 20):
  - Other Urban Transit Systems: 7.81%
  - Residential Property Managers: 7.04%
  - Bus and Other Motor Vehicle Transit Systems: 6.33%
  - Child Day Care Services: 2.38%
  - Hair, Nail, and Skin Care Services: 1.50%
  - Elementary and Secondary Schools: 1.33%
  - Graphic Design Services: 0.88%
  - Offices of Physicians (except Mental Health Specialists): 0.85%
  - Offices of Dentists: 0.70%
  - Religious Organizations: 0.61%
  - Family Planning Centers: 0.50%
  - Environment, Conservation and Wildlife Organizations: 0.36%
  - Malls: 0.31%
  - Custom Computer Programming Services: 0.30%
  - Human Rights Organizations: 0.24%
  - Offices of Chiropractors: 0.23%
  - Civic and Social Organizations: 0.22%
  - Appliance Repair and Maintenance: 0.19%
  - Reupholstery and Furniture Repair: 0.18%
  - Specialized Freight (except Used Goods) Trucking, Long-Distance: 0.18%

SUB_CATEGORY ONLY in US (300, showing top 20):
  - Direct Property and Cas

In [62]:
# Memory cleanup before heavy operations
import gc

# Delete large dataframes we no longer need
if 'df_us' in dir():
    del df_us
if 'df_us_poi' in dir() and len(df_us_poi) > 100000:
    # Keep only what we need for later
    pass  # We still need df_us_poi for crosstab

gc.collect()
print("Memory cleanup complete")

Memory cleanup complete


## 6. Visit Distribution Analysis

In [63]:
# Swedish visits by TOP_CATEGORY (MEMORY-SAFE: aggregate first, then merge)
if df_se_assigned is not None and top_cat_col:
    print("Aggregating Swedish visits by POI (memory-safe approach)...")
    
    # Step 1: Count visits per POI (reduces 80M rows to ~600K)
    matched_stops = df_se_assigned[df_se_assigned['poi_id'].notna()]
    visits_per_poi = matched_stops.groupby('poi_id').size().reset_index(name='visit_count')
    print(f"  Unique POIs with visits: {len(visits_per_poi):,}")
    
    # Step 2: Create POI category lookup (small table)
    poi_id_col = 'PLACEKEY'
    poi_cats = df_se_poi[[poi_id_col, top_cat_col, sub_cat_col]].copy()
    poi_cats.columns = ['poi_id', 'top_category', 'sub_category']
    print(f"  POI category lookup: {len(poi_cats):,} rows")
    
    # Step 3: Merge visits with categories (small merge)
    visits_with_cats = visits_per_poi.merge(poi_cats, on='poi_id', how='left')
    print(f"  Merged: {len(visits_with_cats):,} rows")
    
    # Step 4: Aggregate by category (weighted by visit count)
    se_visit_by_cat = visits_with_cats.groupby('top_category')['visit_count'].sum()
    se_visit_dist = (se_visit_by_cat / se_visit_by_cat.sum()).sort_values(ascending=False)
    
    print(f"\nSwedish VISITS by TOP_CATEGORY (total visits: {se_visit_by_cat.sum():,}):")
    for cat, pct in se_visit_dist.head(15).items():
        print(f"  {cat}: {pct*100:.1f}%")
    
    # Clean up
    del matched_stops, visits_per_poi, visits_with_cats
    import gc; gc.collect()

Aggregating Swedish visits by POI (memory-safe approach)...


  Unique POIs with visits: 514,764
  POI category lookup: 658,731 rows
  Merged: 514,764 rows

Swedish VISITS by TOP_CATEGORY (total visits: 80,820,135):
  Restaurants and Other Eating Places: 6.4%
  Elementary and Secondary Schools: 6.3%
  Urban Transit Systems: 5.9%
  Activities Related to Real Estate: 4.9%
  Personal Care Services: 4.6%
  Other Amusement and Recreation Industries: 3.9%
  Services to Buildings and Dwellings: 3.7%
  Child Day Care Services: 2.6%
  Other Schools and Instruction: 2.5%
  Building Equipment Contractors: 2.5%
  Residential Building Construction: 2.4%
  Other Personal Services: 2.4%
  Printing and Related Support Activities: 2.1%
  Offices of Physicians: 2.0%
  Automobile Dealers: 1.8%


In [64]:
# US visits by TOP_CATEGORY
if 'df_us' in dir() and 'TOP_CATEGORY' in df_us.columns:
    # Check which visit column exists
    visit_col = 'VISIT_COUNTS' if 'VISIT_COUNTS' in df_us.columns else 'RAW_VISIT_COUNTS' if 'RAW_VISIT_COUNTS' in df_us.columns else None
    
    if visit_col:
        print(f"Using visit column: {visit_col}")
        us_visits = df_us[df_us[visit_col].notna()].copy()
        us_visit_sum = us_visits.groupby('TOP_CATEGORY')[visit_col].sum()
        us_visit_dist = (us_visit_sum / us_visit_sum.sum()).sort_values(ascending=False)
        
        print(f"\nUS VISITS by TOP_CATEGORY (total visits: {us_visit_sum.sum():,.0f}):")
        for cat, pct in us_visit_dist.head(15).items():
            print(f"  {cat}: {pct*100:.1f}%")
    else:
        # Fallback: count POI occurrences as proxy for visits
        print("No visit count column found, using POI count as proxy")
        us_visit_dist = df_us['TOP_CATEGORY'].value_counts(normalize=True)
        print(f"\nUS POI distribution by TOP_CATEGORY:")
        for cat, pct in us_visit_dist.head(15).items():
            print(f"  {cat}: {pct*100:.1f}%")

In [65]:
# Compare visit distributions
if 'se_visit_dist' in dir() and 'us_visit_dist' in dir():
    # Convert to DataFrames if they're Series
    se_df = se_visit_dist.reset_index() if hasattr(se_visit_dist, 'reset_index') else pd.DataFrame(se_visit_dist)
    us_df = us_visit_dist.reset_index() if hasattr(us_visit_dist, 'reset_index') else pd.DataFrame(us_visit_dist)
    
    # Standardize column names
    se_df.columns = ['category', 'se_visits'] if len(se_df.columns) == 2 else ['category', 'se_visits']
    us_df.columns = ['category', 'us_visits'] if len(us_df.columns) == 2 else ['category', 'us_visits']
    
    visit_compare = pd.merge(se_df, us_df, on='category', how='outer').fillna(0)
    visit_compare['diff'] = visit_compare['se_visits'] - visit_compare['us_visits']
    visit_compare['abs_diff'] = visit_compare['diff'].abs()
    visit_compare = visit_compare.sort_values('abs_diff', ascending=False)
    
    print("="*80)
    print("VISIT DISTRIBUTION COMPARISON (by TOP_CATEGORY)")
    print("="*80)
    print(f"{'Category':<45} {'Sweden %':>10} {'US %':>10} {'Diff':>10}")
    print("-"*80)
    for _, row in visit_compare.iterrows():
        print(f"{str(row['category'])[:45]:<45} {row['se_visits']*100:>9.1f}% {row['us_visits']*100:>9.1f}% {row['diff']*100:>+9.1f}%")
else:
    print("Visit distributions not available. Run cells 27-28 first.")

Visit distributions not available. Run cells 27-28 first.


In [66]:
# Free memory from large assigned stops dataframe
if 'df_se_assigned' in dir():
    del df_se_assigned
    import gc
    gc.collect()
    print("Released memory from df_se_assigned")

Released memory from df_se_assigned


## 7. Detailed Category Cross-tabulation

In [67]:
# TOP x SUB category combinations in Sweden
if top_cat_col and sub_cat_col:
    se_crosstab = df_se_poi.groupby([top_cat_col, sub_cat_col]).size().reset_index(name='count')
    se_crosstab['pct'] = se_crosstab['count'] / se_crosstab['count'].sum() * 100
    se_crosstab = se_crosstab.sort_values('count', ascending=False)
    
    print(f"Swedish TOP x SUB combinations ({len(se_crosstab)} unique):")
    print(se_crosstab.head(30).to_string(index=False))

Swedish TOP x SUB combinations (478 unique):
                             TOP_CATEGORY                                                     SUB_CATEGORY  count      pct
                    Urban Transit Systems                                      Other Urban Transit Systems  49392 7.814646
        Activities Related to Real Estate                                    Residential Property Managers  44471 7.036061
                    Urban Transit Systems                      Bus and Other Motor Vehicle Transit Systems  40005 6.329464
      Services to Buildings and Dwellings                                             Landscaping Services  28134 4.451272
      Restaurants and Other Eating Places                                         Full-Service Restaurants  23265 3.680915
        Residential Building Construction                                           Residential Remodelers  20443 3.234427
                  Child Day Care Services                                          Child Day C

In [68]:
# TOP x SUB category combinations in US
if 'df_us_poi' in dir():
    us_crosstab = df_us_poi.groupby(['TOP_CATEGORY', 'SUB_CATEGORY']).size().reset_index(name='count')
    us_crosstab['pct'] = us_crosstab['count'] / us_crosstab['count'].sum() * 100
    us_crosstab = us_crosstab.sort_values('count', ascending=False)
    
    print(f"US TOP x SUB combinations ({len(us_crosstab)} unique):")
    print(us_crosstab.head(30).to_string(index=False))

US TOP x SUB combinations (616 unique):
                                                          TOP_CATEGORY                                                    SUB_CATEGORY  count      pct
                                   Restaurants and Other Eating Places                                        Full-Service Restaurants 259157 9.742074
                                   Restaurants and Other Eating Places                                     Limited-Service Restaurants 123038 4.625171
                                   Other Miscellaneous Store Retailers All Other Miscellaneous Store Retailers (except Tobacco Stores) 122666 4.611187
                                                     Gasoline Stations                       Gasoline Stations with Convenience Stores 109020 4.098214
                                                        Grocery Stores      Supermarkets and Other Grocery (except Convenience) Stores  72283 2.717219
                           Activities Related to Credi

## 8. Visualizations

In [69]:
# Plot visit distribution comparison (if available)
if 'visit_compare' in dir() and len(visit_compare) > 0:
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Get top 15 categories
    visit_compare['total'] = visit_compare['se_visits'] + visit_compare['us_visits']
    top15_visits = visit_compare.nlargest(15, 'total').copy()
    
    x = range(len(top15_visits))
    width = 0.35
    
    ax.bar([i - width/2 for i in x], top15_visits['se_visits']*100, width, label='Sweden', color='steelblue')
    ax.bar([i + width/2 for i in x], top15_visits['us_visits']*100, width, label='US', color='coral')
    
    ax.set_ylabel('Percentage of Visits')
    ax.set_title('Visit Distribution by TOP_CATEGORY: Sweden vs US')
    ax.set_xticks(x)
    ax.set_xticklabels(top15_visits['category'], rotation=45, ha='right')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Visit comparison data not available. Run section 6 first.")

Visit comparison data not available. Run section 6 first.


In [70]:
# Plot visit distribution comparison
if 'visit_compare' in dir():
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Get top 15 categories
    visit_compare['total'] = visit_compare['se_visits'] + visit_compare['us_visits']
    top15_visits = visit_compare.nlargest(15, 'total')
    
    x = range(len(top15_visits))
    width = 0.35
    
    ax.bar([i - width/2 for i in x], top15_visits['se_visits']*100, width, label='Sweden', color='steelblue')
    ax.bar([i + width/2 for i in x], top15_visits['us_visits']*100, width, label='US', color='coral')
    
    ax.set_ylabel('Percentage of Visits')
    ax.set_title('Visit Distribution by TOP_CATEGORY: Sweden vs US')
    ax.set_xticks(x)
    ax.set_xticklabels(top15_visits.index, rotation=45, ha='right')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

## 9. Summary & Recommendations

In [71]:
# Summary statistics
print("="*80)
print("SUMMARY")
print("="*80)

if 'df_se_poi' in dir():
    print(f"\nSweden:")
    print(f"  Total POIs: {len(df_se_poi):,}")
    if top_cat_col:
        print(f"  TOP_CATEGORY count: {df_se_poi[top_cat_col].nunique()}")
    if sub_cat_col:
        print(f"  SUB_CATEGORY count: {df_se_poi[sub_cat_col].nunique()}")

if 'df_us_poi' in dir():
    print(f"\nUS:")
    print(f"  Total POIs: {len(df_us_poi):,}")
    if 'TOP_CATEGORY' in df_us_poi.columns:
        print(f"  TOP_CATEGORY count: {df_us_poi['TOP_CATEGORY'].nunique()}")
    if 'SUB_CATEGORY' in df_us_poi.columns:
        print(f"  SUB_CATEGORY count: {df_us_poi['SUB_CATEGORY'].nunique()}")

if 'top_compare' in dir():
    se_only_count = len(top_compare[top_compare['us_pct'] == 0])
    us_only_count = len(top_compare[top_compare['se_pct'] == 0])
    shared_count = len(top_compare) - se_only_count - us_only_count
    
    print(f"\nTOP_CATEGORY Alignment:")
    print(f"  Shared categories: {shared_count}")
    print(f"  Sweden-only: {se_only_count}")
    print(f"  US-only: {us_only_count}")

if 'sub_compare' in dir():
    se_only_sub_count = len(sub_compare[sub_compare['us_pct'] == 0])
    us_only_sub_count = len(sub_compare[sub_compare['se_pct'] == 0])
    shared_sub_count = len(sub_compare) - se_only_sub_count - us_only_sub_count
    
    print(f"\nSUB_CATEGORY Alignment:")
    print(f"  Shared categories: {shared_sub_count}")
    print(f"  Sweden-only: {se_only_sub_count}")
    print(f"  US-only: {us_only_sub_count}")

SUMMARY

Sweden:
  Total POIs: 658,731
  TOP_CATEGORY count: 242
  SUB_CATEGORY count: 478

US:
  Total POIs: 2,661,032
  TOP_CATEGORY count: 235
  SUB_CATEGORY count: 616

TOP_CATEGORY Alignment:
  Shared categories: 183
  Sweden-only: 59
  US-only: 52

SUB_CATEGORY Alignment:
  Shared categories: 316
  Sweden-only: 162
  US-only: 300


In [72]:
# Export comparison tables
if 'top_compare' in dir():
    top_compare.to_csv('dbs/us_foot_traffic/top_category_comparison.csv', index=False)
    print("Saved: dbs/us_foot_traffic/top_category_comparison.csv")

if 'sub_compare' in dir():
    sub_compare.to_csv('dbs/us_foot_traffic/sub_category_comparison.csv', index=False)
    print("Saved: dbs/us_foot_traffic/sub_category_comparison.csv")

Saved: dbs/us_foot_traffic/top_category_comparison.csv
Saved: dbs/us_foot_traffic/sub_category_comparison.csv


## 10. Category Mapping Recommendations

Based on the comparison above, create a unified category schema.

In [73]:
# Potential unified schema (to be refined based on results above)
UNIFIED_CATEGORIES = {
    # Mapping: SafeGraph TOP_CATEGORY -> Unified Category
    'Restaurants and Other Eating Places': 'Food & Dining',
    'Drinking Places (Alcoholic Beverages)': 'Food & Dining',
    'Grocery Stores': 'Retail - Grocery',
    'Supermarkets and Other Grocery (except Convenience) Stores': 'Retail - Grocery',
    'Gasoline Stations': 'Retail - Auto',
    'Clothing Stores': 'Retail - Shopping',
    'Department Stores': 'Retail - Shopping',
    'General Merchandise Stores, including Warehouse Clubs and Supercenters': 'Retail - Shopping',
    'Health and Personal Care Stores': 'Retail - Health',
    'Sporting Goods, Hobby, and Musical Instrument Stores': 'Retail - Specialty',
    'Hotels (except Casino Hotels) and Motels': 'Accommodation',
    'Fitness and Recreational Sports Centers': 'Recreation',
    'Offices of Physicians': 'Healthcare',
    'Elementary and Secondary Schools': 'Education',
    'Religious Organizations': 'Religious',
    'Banks': 'Financial',
    # Add more based on actual data...
}

print("Draft unified category schema:")
for orig, unified in list(UNIFIED_CATEGORIES.items())[:10]:
    print(f"  {orig[:50]} -> {unified}")

Draft unified category schema:
  Restaurants and Other Eating Places -> Food & Dining
  Drinking Places (Alcoholic Beverages) -> Food & Dining
  Grocery Stores -> Retail - Grocery
  Supermarkets and Other Grocery (except Convenience -> Retail - Grocery
  Gasoline Stations -> Retail - Auto
  Clothing Stores -> Retail - Shopping
  Department Stores -> Retail - Shopping
  General Merchandise Stores, including Warehouse Cl -> Retail - Shopping
  Health and Personal Care Stores -> Retail - Health
  Sporting Goods, Hobby, and Musical Instrument Stor -> Retail - Specialty
